# 03 - Incorporacion de anios y benchmark (Ejercicios 5, 6 y 8.3)

Parte 1: validar que 2024, 2025 y 2026 se consultan juntos **con las mismas consultas** (5.6-5.8, 8.3).
Parte 2: analizar `docs/benchmark_results.csv` (6.7-6.9).

In [ ]:
import os, sys
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60, "display.width", 180, "display.max_colwidth", 60)

# Dentro del contenedor el proyecto esta en /workspace; fuera, en la carpeta padre de notebooks/.
RAIZ = Path("/workspace") if Path("/workspace/sql").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
import taxi_common as tc
DATOS = tc.data_dir().as_posix()          # carpeta data/ segun donde se ejecute
from document_queries import abrir

# FUENTE = "parquet": la vista `trips` lee los Parquet de data/raw (normalizados por taxi_common.py).
# FUENTE = "tabla"  : la vista `trips` apunta a la tabla de data/processed/taxi.duckdb (abierta en SOLO LECTURA).
# Las MISMAS consultas sirven para ambas fuentes. Para cambiar: LAB_FUENTE=tabla antes de iniciar Jupyter.
FUENTE = os.environ.get("LAB_FUENTE", "parquet")
con = abrir(FUENTE, str(tc.db_por_defecto()))

def sql(archivo: str) -> pd.DataFrame:
    """Ejecuta un archivo de sql/ y devuelve un DataFrame. La consulta vive en el .sql, no en el notebook."""
    texto = (RAIZ / "sql" / archivo).read_text(encoding="utf-8").replace("/workspace/data", DATOS)
    return con.execute(texto).df()

print("DuckDB", duckdb.__version__, "| fuente:", FUENTE, "| archivos:", len(tc.listar_archivos()))

## Parte 1 - Consulta conjunta de los anios

## 5.6 Consulta conjunta directamente sobre Parquet

Un comodin cubre todos los anios; no hay que listar archivos.

Consulta: `sql/03_validacion_anios/v00_consulta_conjunta_parquet.sql`

In [ ]:
sql("03_validacion_anios/v00_consulta_conjunta_parquet.sql")

### Viajes por archivo (anio y mes)

`sql/03_validacion_anios/v01_viajes_por_anio_mes.sql`

In [ ]:
sql("03_validacion_anios/v01_viajes_por_anio_mes.sql")

### Meses faltantes (vacio = sin huecos; un mes final ausente es normal por el atraso de la TLC)

`sql/03_validacion_anios/v02_meses_faltantes.sql`

In [ ]:
sql("03_validacion_anios/v02_meses_faltantes.sql")

### Fechas fuera del mes de su archivo

`sql/03_validacion_anios/v03_fechas_fuera_de_su_archivo.sql`

In [ ]:
sql("03_validacion_anios/v03_fechas_fuera_de_su_archivo.sql")

### Columnas que cambian entre anios

`sql/03_validacion_anios/v04_columnas_por_anio.sql`

In [ ]:
sql("03_validacion_anios/v04_columnas_por_anio.sql")

### Metricas por anio

`sql/03_validacion_anios/v05_metricas_por_anio.sql`

In [ ]:
sql("03_validacion_anios/v05_metricas_por_anio.sql")

### Variacion interanual del mismo mes

`sql/03_validacion_anios/v06_comparacion_mismo_mes.sql`

In [ ]:
sql("03_validacion_anios/v06_comparacion_mismo_mes.sql")

### Control: filas de `trips` vs footer de cada Parquet (debe estar vacio)

`sql/03_validacion_anios/v08_filas_vs_parquet.sql`

In [ ]:
sql("03_validacion_anios/v08_filas_vs_parquet.sql")

### Control: `trips` vs `ingest_log` (solo con la tabla; vacio = coincide)

`sql/03_validacion_anios/v07_filas_vs_ingest_log.sql`

In [ ]:
sql("03_validacion_anios/v07_filas_vs_ingest_log.sql") if FUENTE == "tabla" else "Requiere LAB_FUENTE=tabla"

### 5.7 ¿Hubo que modificar las consultas?

<!-- COMPLETAR tras agregar 2024 y 2025: ver docs/validacion_anios.md -->

## Parte 2 - Benchmark: Parquet directo vs tabla materializada

Requiere `python scripts/benchmark.py`.

In [ ]:
res = pd.read_csv(RAIZ / "docs" / "benchmark_results.csv")
res.head()

### Razon Parquet / Tabla por consulta y escala (> 1: la tabla fue mas rapida)

In [ ]:
t = res.pivot_table(index=["escala_meses", "filas"], columns=["consulta", "fuente"], values="mediana_s")
razon = pd.DataFrame({q: t[(q, "parquet")] / t[(q, "tabla")] for q in sorted({q for q, _ in t.columns})}).round(2)
razon

### Tiempo vs cantidad de datos

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
for ax, (q, g) in zip(axes.flat, res.groupby("consulta")):
    for fuente, gg in g.groupby("fuente"):
        ax.plot(gg["filas"] / 1e6, gg["mediana_s"], marker="o", label=fuente)
    ax.set_title(q, fontsize=9); ax.set_xlabel("millones de filas"); ax.set_ylabel("mediana (s)")
axes.flat[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
print("Resultados distintos entre estrategias:", int((~res["resultado_equivalente"]).sum()))

El analisis escrito esta en `docs/benchmark_analisis.md`.